# Recommendation system

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [61]:
import pandas as pd
df = pd.read_csv("../data/processed/clean_data.csv")

print(df.shape)

(1726, 6)


In [62]:
df.isnull().sum()

title           0
tags_x          0
genre           0
popularity      0
vote_count      0
vote_average    0
dtype: int64

In [63]:
df.duplicated().sum()

np.int64(0)

In [64]:
df.iloc[0]['genre']

'Animation, Family, Fantasy, Comedy, Music, Adventure'

# NLP text preprocessing

firstly remove punctuations

In [65]:
df['tags_x'][1]

"theodor melfi melissa mccarthy, chri o'dowd, kevin kline, timothi olyphant, dave diggs, skyler gisondo, laura harrier, rosalind chao, kimberli quinn, loretta devine, ravi kapoor comedies, drama a woman adjust to life after a loss contend with a feisti bird that' taken over her garden — and a husband who' struggl to find a way forward."

In [66]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [67]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thaku\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\thaku\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [68]:
stopwords = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [69]:
def preprocess_text(text):
    #change in lowercase
    text = str(text).lower()
    #remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '',text)
    #tokennization
    words = text.split()
    #remove stopwords
    words = [word for word in words if word not in stopwords]
    #lemmatizer
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)    

In [70]:
df['tags_x'] = df['tags_x'].apply(preprocess_text)

In [71]:
df['tags_x'][1]

'theodor melfi melissa mccarthy chri odowd kevin kline timothi olyphant dave diggs skyler gisondo laura harrier rosalind chao kimberli quinn loretta devine ravi kapoor comedy drama woman adjust life loss contend feisti bird taken garden husband struggl find way forward'

# text vectorization

In [72]:
df = df.reset_index(drop = True)

In [73]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

title
my little pony: a new generation       0
the starling                           1
je suis karl                           2
confessions of an invisible girl       3
intrusion                              4
                                    ... 
young adult                         1721
yours, mine and ours                1722
zodiac                              1723
zombieland                          1724
zoom                                1725
Length: 1726, dtype: int64

# TF-IDF method

In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['tags_x'])

# cosine similarity

In [75]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

print(similarity.shape)

(1726, 1726)


In [76]:
def recommend(title):

    title = title.lower()

    if title not in indices:
        print("Movie not found")
        return

    idx = indices[title]

    sim_score = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    movie_indices = sim_score.argsort()[-6:-1][::-1]

    for i in movie_indices:
        print(df.iloc[i]['title'])

In [77]:
recommend('my little pony: a new generation')

over the moon
bombshell: the hedy lamarr story
hop
the willoughbys
turbo


In [78]:
import os
os.makedirs("models", exist_ok=True)

In [79]:
import pickle

# Save TF-IDF matrix
pickle.dump(
    tfidf_matrix,
    open("../Models/tfidf_matrix.pkl", "wb")
)

# Save indices
pickle.dump(
    indices,
    open("../Models/indices.pkl", "wb")
)

# Save dataframe
df.to_pickle(
    "../Models/df.pkl"
)

# Save TF-IDF vectorizer/model
pickle.dump(
    tfidf,
    open("../Models/tfidf.pkl", "wb")
)

print("All files saved successfully in Models folder")

All files saved successfully in Models folder


In [80]:
import os
print(os.listdir("models"))

[]


In [81]:
len(tfidf.vocabulary_)

5000

In [82]:
with open(
    r"C:\hybrid_recom_project\smart_hybrid_netflix_recommender\Data\Raw\mymoviedb.csv",
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    for i in range(5):
        print(f.readline())

Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url

2021-12-15,Spider-Man: No Way Home,"Peter Parker is unmasked and no longer able to separate his normal life from the high-stakes of being a super-hero. When he asks for help from Doctor Strange the stakes become even more dangerous, forcing him to discover what it truly means to be Spider-Man.",5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg

2022-03-01,The Batman,"In his second year of fighting crime, Batman uncovers corruption in Gotham City that connects to his own family while facing a serial killer known as the Riddler.",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R36Fpooo50r9T25onhq.jpg

2022-02-25,No Exit,"Stranded at a rest stop in the mountains during a blizzard, a recovering addict discovers a kidnapped child hidden in a car belonging to one of the people

In [83]:
print(os.listdir("../models"))

['df.pkl', 'hybrid.pkl', 'indices.pkl', 'movies.pkl', 'similarity.pkl', 'tfidf.pkl', 'tfidf_matrix.pkl', 'user_movie_list.pkl', 'user_movie_matrix.pkl', 'user_similarity.pkl']


In [84]:
print(df.columns.tolist())

['title', 'tags_x', 'genre', 'popularity', 'vote_count', 'vote_average']


In [85]:
print('avatar' in indices.index)

False


In [86]:
recommend("avatar")

Movie not found
